# 01 · Descarga y exploración de SECOP II

**Qué hace:** descarga los contratos de la API de datos.gov.co por año y mes, los guarda
como CSV en `datos/raw/contratos/anio=AAAA/`, compara cuántas filas tenemos contra
cuántas dice el servidor y explora el esquema.

**Por qué así** (la lógica está en `src/secop/descarga.py`, probada sin internet):
- **Por mes:** si la API se cae a mitad, solo se repite un mes y no todo el año.
- **Reanudable:** cada página se guarda como archivo y, al terminar un mes, se crea un
  archivo vacío `mes_MM.listo`. Si el notebook se interrumpe, al volver a ejecutarlo
  salta lo que ya existe.
- **Reintentos con espera creciente:** la API a veces responde lento o con error
  5xx/429. Se espera 5 s, 10 s, 20 s... antes de reintentar.
- **Sin datos personales:** se pide a la API solo la lista de columnas permitidas
  (`$select`), así nombres, documentos y cuentas bancarias nunca llegan al disco.

La carpeta `datos/` está en `.gitignore`: los CSV **nunca** se suben a GitHub.

Este notebook es autocontenido: no depende de variables de otros notebooks.

## Configuración: importar `src/secop` y crear la sesión de Spark

1. Buscamos la carpeta del repositorio (la que contiene `src/secop`) subiendo desde la
   carpeta del notebook y añadimos `src` a `sys.path`, la lista de carpetas donde Python
   busca módulos. Así `import secop` funciona sin instalar nada.
2. `entorno.crear_spark()` crea una sesión de **Spark en modo local** con Delta Lake: el
   mismo motor que en un clúster, usando los núcleos del Codespace.

Si cambias código en `src/`, reinicia el kernel (botón *Restart*) para que Python lo
vuelva a cargar.

In [ ]:
import sys
from pathlib import Path


def buscar_raiz_repo() -> Path:
    """Devuelve la carpeta raíz del repo (la que contiene src/secop)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "src" / "secop").is_dir():
            return carpeta
    raise RuntimeError("No encontré src/secop: abre el notebook desde la carpeta del repositorio.")


RAIZ_REPO = buscar_raiz_repo()
if str(RAIZ_REPO / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ_REPO / "src"))

import secop
from secop import config, entorno

spark = entorno.crear_spark()
print("secop", secop.__version__, "| Spark", spark.version, "| datos en", config.raiz_datos())

## 1. Parámetros
- `ANIOS`: años a descargar, separados por coma.
- `MODO`: `descargar` (descarga lo que falte y luego verifica) o `solo_verificar`.

Se pueden cambiar aquí o con variables de entorno (`SECOP_ANIOS`, `SECOP_MODO`).

In [ ]:
ANIOS = [int(a) for a in entorno.parametro("ANIOS", "2019,2020").split(",") if a.strip()]
MODO = entorno.parametro("MODO", "descargar")
assert MODO in ("descargar", "solo_verificar"), MODO
print("Años:", ANIOS, "| modo:", MODO)

## 2. Descargar
Los meses con `mes_MM.listo` se saltan. La primera descarga de un año puede tardar
bastante (la API entrega 50.000 filas por página y a veces responde lento).

In [ ]:
from secop import descarga

if MODO == "descargar":
    for anio in ANIOS:
        for mes in range(1, 13):
            descarga.descargar_mes(anio, mes)
else:
    print("Modo solo_verificar: no se descarga nada.")

## 3. Verificar: filas locales vs. servidor

Leemos los CSV con Spark. Opciones importantes:
- `pathGlobFilter="*.csv"`: ignora los marcadores `.listo` y archivos `.tmp`.
- `multiLine=True` y `escape='"'`: algunas descripciones tienen saltos de línea y comillas
  dentro del texto; sin esto una fila se partiría en varias.
- Sin `inferSchema`: todo se lee como texto (los tipos se corrigen en plata).

`count()` es una **acción**: es el momento en que Spark realmente lee los archivos.
Valores de referencia de la descarga previa: 2019 → 142.090 filas; 2020 → 356.623.

In [ ]:
from pyspark.sql import functions as F


def leer_csv_anio(anio: int):
    return (
        spark.read.option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .option("pathGlobFilter", "*.csv")
        .csv(config.ruta_anio(anio))
    )


filas = []
for anio in ANIOS:
    n_local = leer_csv_anio(anio).count()
    n_servidor = descarga.conteo_servidor(descarga.filtro_anio(anio))
    filas.append((anio, n_local, n_servidor, n_local - n_servidor))

entorno.mostrar(
    spark.createDataFrame(
        filas, "anio int, filas_locales long, filas_servidor long, diferencia long"
    )
)

Una diferencia pequeña puede deberse a contratos actualizados en el servidor después de
la descarga. Una diferencia grande indica un mes incompleto: borra su `mes_MM.listo` y
sus partes, y vuelve a ejecutar en modo `descargar`.

Limitación: los contratos sin fecha de firma no se descargan (el filtro es por esa fecha).

## 4. Exploración del esquema
Revisamos que lleguen las columnas esperadas. Solo se exploran columnas permitidas.

In [ ]:
from secop.esquema import COLUMNAS_BRONCE
from secop.validacion import columnas_personales_presentes, comparar_columnas

crudo = leer_csv_anio(ANIOS[0])
resultado = comparar_columnas(crudo.columns, COLUMNAS_BRONCE)
print("Columnas leídas:", len(crudo.columns))
print("¿Están todas las permitidas?", resultado["valido"], "| faltantes:", resultado["faltantes"])
print("Columnas personales en el CSV:", len(columnas_personales_presentes(crudo.columns)))

permitidas = [c for c in crudo.columns if c in COLUMNAS_BRONCE]
explorar = crudo.select(*permitidas)

### Porcentaje de vacíos por columna
Una sola agregación calcula todas las columnas a la vez (una pasada por los datos). El
resultado es una fila; `stack` la convierte en pares (columna, porcentaje).

In [ ]:
vacios = explorar.select(
    [F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 1).alias(c) for c in permitidas]
)
pares = ", ".join(f"'{c}', `{c}`" for c in permitidas)
entorno.mostrar(
    vacios.select(F.expr(f"stack({len(permitidas)}, {pares}) AS (columna, pct_vacios)")).orderBy(
        F.desc("pct_vacios")
    ),
    filas=80,
)

### Valores más frecuentes de columnas clave
Aquí se ven los problemas que se corrigen en plata: mayúsculas inconsistentes en el estado,
"No Definido", duración con formatos mezclados y localización compuesta.

In [ ]:
for columna in [
    "estado_contrato",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "habilita_pago_adelantado",
    "duraci_n_del_contrato",
    "localizaci_n",
]:
    print("\n" + columna)
    entorno.mostrar(explorar.groupBy(columna).count().orderBy(F.desc("count")), filas=15)

### Distribución del valor del contrato
`percentile_approx` calcula percentiles aproximados de forma distribuida (no ordena todo).

In [ ]:
valor = F.col("valor_del_contrato").try_cast("double")
entorno.mostrar(
    explorar.select(
        F.count(valor).alias("con_valor"),
        F.sum((valor <= 0).cast("int")).alias("valores_menores_o_iguales_a_0"),
        F.percentile_approx(valor, [0.5, 0.9, 0.99]).alias("p50_p90_p99"),
        F.max(valor).alias("maximo"),
    )
)